# Mapeando Datos y Fijando Valores
## Dominando Diccionarios y Tuplas en Python

**Semana 09 - Fundamentos de Python**  
**Cuaderno de trabajo del estudiante**  
**Caso:** análisis de hasta 1,000,000 de pacientes sintéticos

En esta práctica trabajamos con diccionarios para organizar conteos y con tuplas para conservar resultados relacionados. Comenzamos con una muestra y, cuando el análisis esté listo, usamos el mismo código con el millón de registros.

Al finalizar podremos:

1. recorrer claves y valores de un diccionario;
2. contar categorías con `.get()`;
3. recorrer una lista interna con ciclos anidados;
4. construir un diccionario que contiene otros diccionarios;
5. representar un resultado mediante una tupla.


## Preparamos y cargamos los datos

El archivo contiene datos clínicos sintéticos. Los nombres iniciales permiten reconocer el conjunto, pero las edades, enfermedades, provincias, medicamentos y síntomas no representan información real.

Iniciamos con una muestra independiente de 5,000 registros. En la Actividad 8 cambiaremos el control para generar y cargar el millón. No abrimos los archivos JSON directamente en el editor.


In [1]:
from pathlib import Path
import sys

carpeta_semana = Path.cwd()
if not (carpeta_semana / "datos_s09.py").exists():
    carpeta_semana = Path.cwd() / "Semana 09"

if not (carpeta_semana / "datos_s09.py").exists():
    raise FileNotFoundError("Abrimos el notebook desde la raíz del curso o desde Semana 09.")

sys.path.insert(0, str(carpeta_semana.resolve()))
from datos_s09 import cargar_datos


In [2]:
USAR_MILLON = False
registros, ruta_datos = cargar_datos(USAR_MILLON, carpeta_semana)

print(f"Registros cargados: {len(registros):,}")
print(f"Archivo utilizado: {ruta_datos.name}")


Registros cargados: 5,000
Archivo utilizado: clinica_s09_muestra.json


## Iniciamos la cadena de análisis

Partimos de `registros`, el conjunto seleccionado en la celda anterior. Cada actividad utilizará resultados de la actividad previa hasta construir un resumen por provincia.


In [3]:
from verificaciones_s09 import (
    verificar_contador, verificar_contador_filtrado, verificar_mapa_anidado,
    verificar_inicio_mapa, verificar_maximo, verificar_totales_por_grupo,
    verificar_tupla,
)

print("Verificaciones de apoyo disponibles.")


Verificaciones de apoyo disponibles.


In [4]:
print("Comenzamos con el primer registro del conjunto seleccionado.")


Comenzamos con el primer registro del conjunto seleccionado.


## Actividad 1 - Exploramos un registro

Seleccionamos el primer paciente de `registros`. Consultamos su enfermedad, provincia y lista de síntomas; estos valores se usarán en las actividades siguientes.


In [5]:
# Seleccionamos el primer registro y exploramos sus tipos, claves y campos.
print("Tipo registro: ", type(registros).__name__)
print("cantidad registro: ", len(registros))

#PRIMER REGISTRO

primer_paciente = registros[0]

print("tipo de paciente:", type(primer_paciente))
print("los campos disponibles:", tuple(primer_paciente.keys()))
print("enfermedad: " , primer_paciente["enfermedad"])
print("provincias: " , primer_paciente["provincia"])
print("sintomas: " , primer_paciente["sintomas"])




Tipo registro:  list
cantidad registro:  5000
tipo de paciente: <class 'dict'>
los campos disponibles: ('carnet', 'nombre', 'edad', 'genero', 'provincia', 'canton', 'visitas', 'enfermedad', 'medicamento', 'activo', 'sintomas')
enfermedad:  alzheimer
provincias:  Alajuela
sintomas:  ['náuseas', 'fatiga', 'dolor muscular']


## Actividad 2 - Convertimos campos en pares y tuplas

Partimos de `primer_paciente`. Creamos `datos_observados` con su enfermedad, provincia y síntomas. Recorremos estos campos con `.items()`, formamos una tupla con la enfermedad y la desempaquetamos para obtener `enfermedad_observada`.


In [6]:
# Creamos datos_observados y mostramos cada asociación al recorrer .items().
# Formamos par_observado con la enfermedad y obtenemos enfermedad_observada.

datos = {
    "enfermedad" : primer_paciente["enfermedad"],
    "localidad" : primer_paciente["provincia"],
    "sintomas"  : primer_paciente["sintomas"]
    
}

for clave, valor in datos.items():
    print("par desempaquetado", clave , "->", valor)

print(datos)

#print("Par desempaquetado:", campo_observado, "->", enfermedad_observada)
#verificar_tupla(par_observado, "el par observado")


par desempaquetado enfermedad -> alzheimer
par desempaquetado localidad -> Alajuela
par desempaquetado sintomas -> ['náuseas', 'fatiga', 'dolor muscular']
{'enfermedad': 'alzheimer', 'localidad': 'Alajuela', 'sintomas': ['náuseas', 'fatiga', 'dolor muscular']}


## Actividad 3 - Construimos un conteo pequeño

Partimos de la enfermedad observada y recorremos solamente `registros[:5]`. Construimos manualmente `conteo_pequeno_enfermedades` para representar cuántas veces aparece cada enfermedad en esos cinco registros.

Usamos `diccionario.get(clave, valor_predeterminado)` para consultar una clave sin producir un error cuando todavía no existe. Si la clave está presente, obtenemos su valor actual; si no está presente, obtenemos el valor predeterminado. En un conteo utilizamos `0` como punto de partida.

Podemos actualizar una categoría con este patrón:

```python
conteo[valor] = conteo.get(valor, 0) + 1
```


In [7]:
conteo_pequeno_enfermedades = {}

# Recorremos los primeros cinco registros y actualizamos el conteo manual.
for paciente in registros:
    clave = paciente["enfermedad"]
    conteo_pequeno_enfermedades[clave] = conteo_pequeno_enfermedades.get(clave,0) + 1
print(conteo_pequeno_enfermedades)

#verificar_contador(
 #   conteo_pequeno_enfermedades, registros[:5], "enfermedad", "conteo pequeño"
#)


{'alzheimer': 118, 'arritmia': 133, 'esclerosis múltiple': 126, 'epilepsia': 133, 'depresión mayor': 134, 'hepatitis': 145, 'ansiedad generalizada': 119, 'otitis': 113, 'psoriasis': 133, 'asma': 134, 'conjuntivitis': 138, 'gota': 116, 'insuficiencia renal': 121, 'osteoporosis': 124, 'cistitis': 108, 'fibromialgia': 116, 'sinusitis': 115, 'migraña': 115, 'hipertiroidismo': 116, 'varicela': 134, 'gripe': 124, 'alergia': 146, 'hipertensión': 134, 'dermatitis': 137, 'trastorno bipolar': 135, 'estrés': 106, 'colitis': 101, 'artritis': 126, 'parkinson': 109, 'diabetes': 136, 'anemia': 139, 'obesidad': 131, 'úlcera péptica': 126, 'bronquitis': 132, 'lupus': 121, 'hipotiroidismo': 123, 'faringitis': 129, 'lumbalgia': 115, 'gastritis': 129, 'tiroiditis': 110}


## Actividad 4 - Generalizamos el conteo en una función

Convertimos el patrón de la Actividad 3 en nuestra única función obligatoria: `contar_por_campo(registros, campo)`. La aplicamos a enfermedad, provincia y medicamento, y consultamos una categoría real de cada conteo.


In [8]:
def contar_por_campo(registros, campo):
    """Cuenta las apariciones de cada valor de un campo."""
    # Generalizamos aquí el patrón del conteo pequeño.
    conteo = {}
    for registro in registros:
        valor = registro[campo]
        conteo[valor] = conteo.get(valor, 0) +1
    return conteo
    
conteo_enfermedades = contar_por_campo(registros, "enfermedad")
conteo_provincias = contar_por_campo(registros, "provincia")
conteo_medicamentos = contar_por_campo(registros, "medicamento")

print(conteo_enfermedades)
print(conteo_provincias)
print(conteo_medicamentos)

# Creamos los tres conteos y consultamos una clave existente de cada uno.

verificar_contador(conteo_enfermedades, registros, "enfermedad", "enfermedades")
verificar_contador(conteo_provincias, registros, "provincia", "provincias")
verificar_contador(conteo_medicamentos, registros, "medicamento", "medicamentos")


{'alzheimer': 118, 'arritmia': 133, 'esclerosis múltiple': 126, 'epilepsia': 133, 'depresión mayor': 134, 'hepatitis': 145, 'ansiedad generalizada': 119, 'otitis': 113, 'psoriasis': 133, 'asma': 134, 'conjuntivitis': 138, 'gota': 116, 'insuficiencia renal': 121, 'osteoporosis': 124, 'cistitis': 108, 'fibromialgia': 116, 'sinusitis': 115, 'migraña': 115, 'hipertiroidismo': 116, 'varicela': 134, 'gripe': 124, 'alergia': 146, 'hipertensión': 134, 'dermatitis': 137, 'trastorno bipolar': 135, 'estrés': 106, 'colitis': 101, 'artritis': 126, 'parkinson': 109, 'diabetes': 136, 'anemia': 139, 'obesidad': 131, 'úlcera péptica': 126, 'bronquitis': 132, 'lupus': 121, 'hipotiroidismo': 123, 'faringitis': 129, 'lumbalgia': 115, 'gastritis': 129, 'tiroiditis': 110}
{'Alajuela': 701, 'San José': 685, 'Heredia': 697, 'Cartago': 714, 'Limón': 759, 'Puntarenas': 745, 'Guanacaste': 699}
{'losartán': 145, 'simvastatina': 130, 'atorvastatina': 133, 'montelukast': 146, 'enalapril': 137, 'cetirizina': 140, 'p

## Actividad 5 - Contamos síntomas con dos ciclos

Extendemos el patrón de conteo a la lista `sintomas` observada en la Actividad 1. Construimos `conteo_sintomas` con un ciclo para los registros y otro para sus síntomas.


In [9]:
conteo_sintomas = {}

# Escribimos aquí los dos ciclos y actualizamos el conteo.
for paciente in registros:
    sintomas = paciente['sintomas'] 
    
    # lista de sintomas
    for sintoma in sintomas:
        conteo_sintomas[sintoma] = conteo_sintomas.get(sintoma, 0) + 1

print(conteo_sintomas)    

verificar_contador(
    conteo_sintomas,
    registros,
    "sintomas",
    "síntomas",
    campo_lista=True,
)


{'náuseas': 209, 'fatiga': 224, 'dolor muscular': 233, 'confusión': 219, 'fiebre': 232, 'dificultad para tragar': 197, 'caída de cabello': 234, 'depresión': 243, 'vómitos': 225, 'mareos': 256, 'debilidad muscular': 224, 'visión borrosa': 232, 'irritabilidad': 241, 'dificultad para respirar': 208, 'dolor abdominal': 230, 'heridas que no sanan': 237, 'estornudos': 213, 'secreción nasal': 232, 'dolor al orinar': 237, 'dolor de cabeza': 231, 'picazón': 240, 'inflamación': 257, 'sudoración nocturna': 249, 'micción frecuente': 217, 'boca seca': 236, 'escalofríos': 248, 'calambres': 227, 'piel grasosa': 222, 'pérdida de apetito': 244, 'esputo amarillo': 223, 'erupciones': 225, 'insomnio': 244, 'zumbido en oídos': 242, 'moretones': 238, 'tos con sangre': 254, 'dolor articular': 232, 'sensibilidad al sonido': 252, 'palpitaciones': 243, 'hormigueo': 223, 'ictericia': 220, 'manchas en la piel': 230, 'dolor de garganta': 250, 'dolor lumbar': 266, 'convulsiones': 244, 'ojos rojos': 229, 'sed excesi

## Actividad 6 - Filtramos síntomas de una provincia

Partimos de la provincia del primer paciente y de los dos ciclos anteriores. Construimos `conteo_sintomas_provincia`, un diccionario plano que incluye únicamente los síntomas de esa provincia.


In [10]:
provincia_filtrada = primer_paciente["provincia"]
conteo_sintomas_provincia = {}

# Filtramos por provincia y contamos sus síntomas con dos ciclos.

for paciente in registros:
    if paciente["provincia"] == provincia_filtrada:
        sintomas = paciente['sintomas'] 
    
    # lista de sintomas
        for sintoma in sintomas:
            conteo_sintomas_provincia[sintoma] = conteo_sintomas_provincia.get(sintoma, 0) + 1

print(conteo_sintomas_provincia)
print("Total global de síntomas:", sum(conteo_sintomas.values()))
print("Total de la provincia:", sum(conteo_sintomas_provincia.values()))


verificar_contador_filtrado(
    conteo_sintomas_provincia, registros, "provincia",
    provincia_filtrada, "sintomas", "síntomas de la provincia",
)


{'náuseas': 31, 'fatiga': 24, 'dolor muscular': 30, 'caída de cabello': 41, 'depresión': 41, 'vómitos': 37, 'heridas que no sanan': 43, 'estornudos': 23, 'inflamación': 39, 'sudoración nocturna': 33, 'micción frecuente': 29, 'piel grasosa': 35, 'insomnio': 32, 'zumbido en oídos': 31, 'dificultad para respirar': 25, 'tos con sangre': 42, 'dolor articular': 37, 'dolor de cabeza': 30, 'sed excesiva': 27, 'rigidez matutina': 40, 'dolor al orinar': 38, 'sangrado': 34, 'piel seca': 32, 'moretones': 30, 'sensibilidad al sonido': 48, 'pérdida de apetito': 38, 'pérdida de equilibrio': 40, 'ansiedad': 38, 'dolor de pecho': 32, 'secreción nasal': 30, 'dolor de garganta': 35, 'dificultad para tragar': 22, 'diarrea': 38, 'pérdida de peso': 22, 'palpitaciones': 29, 'boca seca': 36, 'tos': 27, 'ictericia': 30, 'mareos': 35, 'pérdida de memoria': 29, 'confusión': 26, 'dolor abdominal': 28, 'esputo amarillo': 31, 'mal aliento': 27, 'calambres': 36, 'heces oscuras': 41, 'uñas frágiles': 31, 'dolor lumba

## Actividad 7 - Construimos y consultamos el mapa provincial

Convertimos el filtro de la Actividad 6 en un mapa que conserva los conteos de todas las provincias. Primero creamos un checkpoint con el primer registro y después extendemos el mapa al resto de `registros`:

```python
{provincia: {síntoma: cantidad}}
```

Finalmente recorremos ambos niveles con `.items()`, totalizamos cada provincia y fijamos como tupla el síntoma mayor de `provincia_filtrada`.


### 7A - Iniciamos el mapa con un registro

Partimos de `primer_paciente`. Creamos el primer diccionario provincial con sus tres síntomas y comprobamos que el mapa represente únicamente este registro.


In [40]:
provincias= set()

for provincia in registros:
    provincias.add(provincia["provincia"])

sintomas_por_provincia = {}
conteo_sintomas_provincia = {}
# Iniciamos aquí el mapa con la provincia y los síntomas del primer registro.

for paciente in registros[0:25]:
    for provincia in provincias:
        if paciente["provincia"] == provincia:
            sintomas = paciente['sintomas'] 
    
    # lista de sintomas
            for sintoma in sintomas:
                conteo_sintomas_provincia[sintoma] = conteo_sintomas_provincia.get(sintoma, 0) + 1
            sintomas_por_provincia[provincia] = conteo_sintomas_provincia
        conteo_sintomas_provincia= {}
            
print(provincias)
print(sintomas)
print(conteo_sintomas_provincia)
print(sintomas_por_provincia)


verificar_inicio_mapa(
    sintomas_por_provincia, primer_paciente, "provincia", "sintomas",
    "el mapa inicial",
)


{'Limón', 'Cartago', 'Puntarenas', 'San José', 'Alajuela', 'Guanacaste', 'Heredia'}
['depresión', 'dificultad para tragar', 'tos con sangre']
{}
{'Alajuela': {'dolor de cabeza': 1, 'sudoración nocturna': 1, 'inflamación': 1}, 'San José': {'depresión': 1, 'visión borrosa': 1, 'ictericia': 1}, 'Heredia': {'dolor abdominal': 1, 'debilidad muscular': 1, 'náuseas': 1}, 'Cartago': {'moretones': 1, 'inflamación': 1, 'insomnio': 1}, 'Limón': {'depresión': 1, 'dificultad para tragar': 1, 'tos con sangre': 1}, 'Puntarenas': {'visión borrosa': 1, 'depresión': 1, 'manchas en la piel': 1}, 'Guanacaste': {'mareos': 1, 'calambres': 1, 'sensibilidad al sonido': 1}}


AssertionError: Revisamos el mapa inicial: debe representar únicamente el primer registro.

### 7B - Extendemos el mapa a todos los registros

Partimos del mapa inicial. Recorremos `registros[1:]`, agregamos las provincias que todavía no existen y actualizamos sus conteos de síntomas.


In [ ]:
# Extendemos aquí sintomas_por_provincia con los registros restantes.

assert (
    sintomas_por_provincia[provincia_filtrada]
    == conteo_sintomas_provincia
), "Comprobamos que el grupo provincial coincida con el conteo filtrado de la Actividad 6."
print("Comprobamos que el mapa conserva exactamente el conteo filtrado de la Actividad 6.")

verificar_mapa_anidado(
    sintomas_por_provincia,
    registros,
    "provincia",
    "sintomas",
    "síntomas por provincia (checkpoint completo)",
    campo_valor_lista=True,
)


### 7C - Consultamos los dos niveles del mapa

Recorremos el nivel de provincias y el nivel de síntomas mediante `.items()`. Construimos los totales provinciales y fijamos en una tupla el síntoma mayor de `provincia_filtrada`.


In [ ]:
totales_sintomas_por_provincia = {}
sintoma_mayor_provincia = ("", -1)

# Recorremos ambos niveles, totalizamos y actualizamos la tupla máxima.

verificar_totales_por_grupo(
    totales_sintomas_por_provincia, sintomas_por_provincia, "totales provinciales"
)
verificar_maximo(
    sintoma_mayor_provincia, sintomas_por_provincia[provincia_filtrada],
    "síntoma mayor de la provincia filtrada",
)


## Actividad 8 - Validamos con el millón y concluimos

Si `USAR_MILLON` continúa en `False`, nuestras conclusiones corresponden a la muestra. Para validar con el millón, cambiamos el valor a `True` en **Preparamos y cargamos los datos** y volvemos a ejecutar desde esa celda hasta la Actividad 7.

Escribimos exactamente tres conclusiones: cómo generalizamos el conteo pequeño, cómo el filtro provincial se convirtió en un mapa y cómo consultamos el resumen con el volumen seleccionado.


In [ ]:
conclusion_generalizacion = ""
conclusion_mapa_provincial = ""
conclusion_consulta_millon = ""

# Escribimos tres conclusiones con nuestras propias palabras.

if USAR_MILLON:
    print("Validamos la cadena con 1,000,000 de registros.")
else:
    print("Validamos la cadena con la muestra. Cambiamos USAR_MILLON para validar el millón.")

print(conclusion_generalizacion)
print(conclusion_mapa_provincial)
print(conclusion_consulta_millon)


## Cierre

Comprobamos las estructuras con el conjunto seleccionado. Nuestros diccionarios convierten los registros en conteos consultables y nuestras tuplas mantienen unidos los valores de un resultado.
